# 온도 데이터가 몇 초 간격으로 기록되었는지 확인하기

**담당: 정수진**  ·  관련 Issue: #13 (번호를 채우세요)

## 이 노트북에서 할 일

온도 CSV의 측정 시각(`MEAS_DT`)을 보고, 데이터가 **몇 초마다 한 번씩** 찍혔는지 확인하기 위한 작업입니다.

## 왜 하는지

간격을 모르면 나중에 "1분 평균", "10분 이동평균" 같은 걸 계산할 때 기준을 정할 수 없습니다.
또 간격이 일정하지 않다면 그 자체가 설비나 계측기의 문제 신호일 수 있습니다.

## 진행 방법

1. 아래 칸을 **위에서부터 순서대로** 실행하세요.
2. `# TODO` 라고 적힌 곳을 직접 채우세요. 막히면 디스코드에 물어보세요.
3. 맨 아래 **결과 정리** 칸에 확인한 값을 한국어 문장으로 적으세요. 이게 진짜 결과물입니다.
4. 다 했으면 커밋 전에 맨 마지막 안내를 읽으세요.

> 데이터 파일이 없으면 `data/` 폴더에 CSV 3개를 먼저 넣으세요. 이 파일들은 Git에 올라가지 않습니다(공유받은 자료를 각자 직접 넣습니다).

## 공통 준비

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로를 잡아둡니다.
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
TEMP_CSV   = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV  = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")

pd.set_option("display.max_columns", 50)
print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(REPAIR_CSV))

# 컬럼명과 해석 근거

`TC`, `OP`, `CP`, `Z` 같은 접두어·번호는 컬럼을 읽는 단서이고, 공정 구간의 이름과 배정은 데이터 사전의 설명을 따른다. 이름만으로 센서의 실제 설치 위치나 물리 단위를 확정할 수는 없다.

| 컬럼 | 해석 | 구체적인 판단 근거 | 단위 |
| --- | --- | --- | --- |
| `MEAS_DT` | 계측시각 | `DT`가 날짜·시각을 뜻하고 CSV 값이 날짜·시각으로 파싱된다. 아래 코드에서도 변환 실패가 0건이다. | 날짜·시각 |
| `LOT_NO` | 로트번호 | `LOT`와 `NO`의 조합이며 데이터 사전에서 연속 계측 구간을 로트와 연결한다. 따라서 측정값이 아니라 구간 식별자로 읽는다. | 단위 없음(식별자) |
| `CLN-IN` | 세정조 입측 온도 | `CLN`과 `IN`은 세정조 입측을 가리키는 표기이고, 데이터 사전이 이 지점의 온도라고 설명한다. | 온도 단위 확인 필요 |
| `TC-Z1` / `TC-Z2` | 건조존 1존 / 2존 온도 | `TC` 계열을 온도 계측값으로, `Z1`·`Z2`를 존 번호로 읽는다. 두 존이 건조존에 속한다는 배정은 데이터 사전에 따른다. | 온도 단위 확인 필요 |
| `OP-Z1` / `OP-Z2` | 건조존 1존 / 2존 제어출력 | 같은 존 번호의 `TC`와 짝을 이루는 `OP` 계열이다. 데이터 사전에서 이를 제어출력으로, `Z1`·`Z2`를 건조존으로 설명한다. | 출력 단위 확인 필요 |
| `TC-Z3` / `TC-Z4` | 가열·균열존 1존 / 2존 온도 | `TC`는 온도 계열이고 `Z3`·`Z4`는 전체 설비의 연속 존 번호다. 데이터 사전에서 이 번호를 가열·균열존의 첫 두 존에 배정한다. | 온도 단위 확인 필요 |
| `TC-Z5` / `TC-Z6` | 가열·균열존 3존 / 4존 온도 | 앞 행의 `Z3`·`Z4`에 이어지는 온도 계열이며, 데이터 사전에서 같은 공정 구간의 다음 두 존으로 배정한다. | 온도 단위 확인 필요 |
| `OP-Z3` / `OP-Z4` | 가열·균열존 1존 / 2존 제어출력 | `TC-Z3`·`TC-Z4`와 존 번호가 같아 같은 존의 출력으로 대응된다. 출력의 의미와 공정 배정은 데이터 사전에 따른다. | 출력 단위 확인 필요 |
| `OP-Z5` / `OP-Z6` | 가열·균열존 3존 / 4존 제어출력 | `TC-Z5`·`TC-Z6`와 존 번호가 같아 같은 존의 출력으로 대응된다. 출력의 의미와 공정 배정은 데이터 사전에 따른다. | 출력 단위 확인 필요 |
| `CP-Z4` | 가열·균열존 카본포텐셜 제어값 | `CP`는 데이터 사전의 카본포텐셜 계열이고 `Z4`가 대상 존을 가리킨다. 별도 `CP-Z4M`이 있어 이 컬럼은 기준·제어값으로 구분된다. | 확인 필요 |
| `CP-Z4M` | 가열·균열존 카본포텐셜 모니터값 | `CP-Z4`와 같은 존 번호에 `M`이 추가되어 있고 데이터 사전이 모니터값으로 설명한다. 따라서 기준값과 구별해 읽는다. | 확인 필요 |
| `TC-Z7` / `TC-Z8` | 과시효대 1존 / 2존 온도 | `TC` 온도 계열의 다음 두 존이며, 데이터 사전에서 `Z7`·`Z8`을 과시효대에 배정한다. | 온도 단위 확인 필요 |
| `TC-Z9` | 과시효대 슬러지 제거부 온도 | `TC` 온도 계열의 `Z9`이고 데이터 사전의 세부 명칭이 ‘슬러지 제거부’다. 명칭만으로 실제 설비의 동일 장치 존재까지 검증된 것은 아니다. | 온도 단위 확인 필요 |
| `OP-Z7` / `OP-Z8` | 과시효대 1존 / 2존 제어출력 | `TC-Z7`·`TC-Z8`과 번호가 대응하고, 데이터 사전에서 과시효대의 제어출력으로 분류한다. | 출력 단위 확인 필요 |
| `TC-OUT1` / `TC-OUT2` | 냉각대 1존 / 2존 온도 | `OUT`은 출측 표기이고 `TC`는 온도 계열이다. 데이터 사전에서 이 두 채널을 냉각대의 1·2존으로 배정한다. | 온도 단위 확인 필요 |

이 분석의 시간 간격은 `MEAS_DT` 차이를 **초(s)**로 환산해 표시한다. 원본 CSV에는 온도·제어출력·카본포텐셜의 물리 단위가 명시되어 있지 않으므로, 값의 범위만 보고 °C·% 등을 확정하지 않는다. 단위를 확인하기 전에는 이 컬럼들의 그래프 축과 결과표에 단위를 임의로 붙이지 않는다.

### 1단계 — 온도 데이터를 읽고 시각 컬럼을 날짜 형식으로 바꾸세요

CSV에서 읽으면 `MEAS_DT`는 그냥 글자(문자열)입니다.
글자끼리는 빼기를 할 수 없으니, 먼저 **날짜/시간 형식으로 바꿔야** 간격을 계산할 수 있습니다.

In [ ]:
# read_csv는 CSV를 표 형태의 DataFrame으로 읽습니다. 한글이 있으므로 UTF-8 인코딩을 지정합니다.
df = pd.read_csv(TEMP_CSV, encoding="utf-8")

# MEAS_DT를 날짜/시간 형식으로 변환합니다. 변환할 수 없는 값은 NaT가 됩니다.
# errors='coerce'를 사용하면 형식이 잘못된 값이 오류를 중단시키지 않고 NaT(날짜 결측값)로 바뀝니다.
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="coerce")

print(df.shape)       # (행 수, 열 수)
print(df.dtypes.head())  # MEAS_DT가 datetime64로 바뀌었는지 확인합니다.
print("시각 변환 실패:", df["MEAS_DT"].isna().sum(), "건")
print("측정 기간:", df["MEAS_DT"].min(), "~", df["MEAS_DT"].max())

### 2단계 — 시각을 순서대로 정렬하고, 바로 앞 행과의 시간 차이를 구하세요

파일에 적힌 순서가 시간 순서라는 보장이 없습니다. 먼저 정렬하세요.
그다음 `.diff()` 를 쓰면 **바로 위 행과의 차이**를 구할 수 있습니다.

In [ ]:
# LOT 사이의 장기 비가동 시간이 측정 간격에 포함되지 않도록 LOT와 시각으로 정렬합니다.
# sort_values는 LOT를 먼저 묶고 각 LOT 안에서 측정 시각이 빠른 순서로 배치합니다.
# reset_index(drop=True)는 정렬 전 행 번호를 버리고 0부터 새 행 번호를 붙입니다.
df = df.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

# 같은 LOT 내부에서만 바로 앞 측정과의 시간 차이를 계산합니다.
# groupby는 LOT별로 데이터를 나누고, diff는 현재 시각에서 같은 LOT의 직전 시각을 뺍니다.
# 각 LOT의 첫 행은 비교할 이전 행이 없으므로 결과가 NaT가 됩니다.
gap = df.groupby("LOT_NO")["MEAS_DT"].diff()


# gap 은 "0 days 00:00:01" 같은 형태입니다. 초 단위 숫자로 바꾸면 보기 편합니다.
# total_seconds는 시간 차이를 초 단위 실수로 바꾸고, dropna는 LOT별 첫 행을 계산에서 제외합니다.
gap_sec = gap.dt.total_seconds().dropna()

# 0초는 중복 시각, 음수는 시간 역순을 의미하므로 함께 점검합니다.
print("계산한 LOT 내부 간격:", f"{len(gap_sec):,}", "개")
print("0초 간격:", int((gap_sec == 0).sum()), "건")
print("음수 간격:", int((gap_sec < 0).sum()), "건")

### 3단계 — 어떤 간격이 몇 번 나왔는지 세어 보세요

`value_counts()` 는 값별 등장 횟수를 세어줍니다.
가장 위에 나오는 값이 **가장 흔한 간격**입니다.

In [ ]:
# 간격별 발생 건수와 전체 대비 비율을 한 표로 확인합니다.
# value_counts는 같은 간격이 몇 번 등장했는지 셉니다. sort_index는 1초, 2초 순으로 정렬합니다.
# normalize=True이면 건수 대신 0~1 비율을 반환하므로 100을 곱해 백분율로 만듭니다.
interval_summary = pd.DataFrame({
    "count": gap_sec.value_counts().sort_index(),
    "pct": (gap_sec.value_counts(normalize=True).sort_index() * 100).round(4),
})
display(interval_summary)

# ✅ 확인부탁드립니다
⚠️ 우선 3초 이상을 상세 점검 기준으로 잡을 것인지 기준을 잡아야 될 것 같습니다.
이 데이터에서는 2초 간격이 10,372건으로 흔하지만, 3초 이상은 38건이라 발생 시점을 하나씩 살펴볼 수 있습니다.

| LOT 내부 간격 | 우선 처리 방향 |
| --- | --- |
| 2초 | LOT별 건수와 비율을 기록하고, 현재는 정상·이상으로 판정하지 않기 |
| 3초 이상 | 해당 시점의 앞뒤 측정값과 조업 이벤트를 대조하기 |
| 5초 이상 | 7건이므로 우선적으로 개별 원인 확인하기 |

LOT 전체에 ‘이상’ 표시를 붙이는 기준으로 쓰지 않고, 예를 들어 11초 간격이 한 번 있는 LOT와 2초 간격이 반복되는 LOT는 양상이 다릅니다. 
각 LOT에 3초 이상 발생 건수, 1초 초과 비율, 최대 간격을 함께 적고 비교하는 방향이 좋을 것 같습니다.

위 lot 을 EDA 점검 우선순위로 지정하여 데이터를 제외하거나 보간할 기준은 데이터사전에 기록된 정상 수집 주기를 확인한 뒤 정해봅시다.

### 4단계 — 한 문장으로 정리해 보세요

예시: "온도 데이터는 대부분 1초 간격으로 기록되었고(전체의 99.2%), 가끔 2초 이상 벌어진 구간이 있다."

아래 칸에서 숫자를 직접 확인해서 문장을 만들어 보세요.

In [ ]:
# 가장 흔한 간격과 비율을 자동으로 계산하여 한 문장으로 출력합니다.
# mode는 최빈값을 Series로 반환하며 iloc[0]으로 첫 번째 최빈값을 가져옵니다.
most_common_sec = gap_sec.mode().iloc[0]
# 비교 결과인 True/False에서 True를 합산하면 대표 간격의 발생 건수가 됩니다.
most_common_count = int((gap_sec == most_common_sec).sum())
# 불리언의 평균은 True의 비율이므로 100을 곱해 백분율로 변환합니다.
most_common_pct = (gap_sec == most_common_sec).mean() * 100
# 대표 간격이 아닌 값만 고른 뒤 unique로 중복을 제거하고 작은 값부터 정렬합니다.
other_intervals = sorted(gap_sec[gap_sec != most_common_sec].unique().tolist())
print(
    f"온도 데이터는 LOT 내부에서 대부분 {most_common_sec:g}초 간격으로 기록되었고 "
    f"({most_common_count:,}건, {most_common_pct:.2f}%), "
    f"다른 간격은 {other_intervals}초입니다."
)

## 결과 정리 — 여기를 꼭 채우세요

아래 빈칸을 확인한 값으로 바꿔서 적으세요. 이 내용이 `docs/02-data-contract.md`로 옮겨집니다.

| 항목 | 확인한 값 |
| --- | --- |
| 가장 흔한 기록 간격 | 1초 |
| 그 간격이 전체의 몇 % 인지 | 96.99% (334,966 / 345,376개 LOT 내부 간격) |
| 다른 간격도 있었는지 | 있음 — 2초, 3초, 5초, 6초, 10초, 11초 |
| 전체 행 수 | 345,390행 |


### 이상하다고 느낀 점 / 확실하지 않은 점

- LOT 사이에는 장기 비가동 시간이 있으므로 같은 `LOT_NO` 내부에서만 측정 간격을 계산했습니다.
- 2초 간격이 10,372건입니다. 실제 수집 누락인지 허용된 로거 동작인지 설비 사양 확인이 필요합니다.
- LOT 내부 최대 측정 간격은 11초입니다.

---

## 커밋하기 전에 읽으세요

1. **출력은 지우지 않아도 됩니다.** `nbstripout` 필터를 등록해 뒀다면 `git add` 할 때 자동으로 지워집니다.
   등록했는지 확인: `python -m nbstripout --status` → `Automatic cleanup enabled` 가 나와야 합니다.
   안 나오면: `python -m nbstripout --install --attributes .gitattributes`
2. 이 노트북 **한 파일만** 커밋하세요. 다른 사람 파일은 건드리지 마세요.
3. 브랜치를 만들어서 작업하세요. 예) `git switch -c feat/이슈번호-설명`
4. 커밋 메시지 첫 줄: `feat: 온도 데이터 기록 간격 확인`
5. PR 본문에 `Closes #이슈번호` 를 꼭 적으세요.

---

# 추가 점검 및 시각화 — 기본 계산 이후

위 단계에서 대표 간격을 계산한 뒤, 결과가 일부 LOT나 특정 시점에 치우친 것은 아닌지 추가로 점검합니다. 아래 시각화는 숫자 하나만 보고 놓칠 수 있는 불규칙 간격의 크기·LOT별 집중도·발생 시점을 확인하기 위해 추가했습니다.

In [ ]:
# [목적] CSV를 다시 읽어 이 셀만 실행해도 측정 간격 결과를 얻을 수 있게 합니다.
# [확인] 시각 변환 실패가 0건이고 MEAS_DT가 datetime64 자료형인지 확인합니다.
import matplotlib.pyplot as plt
from IPython.display import display

# 기존 df와 구분되는 이름으로 다시 읽어 이 추가 셀의 계산이 앞쪽 변수 변경에 영향을 덜 받게 합니다.
sampling_df = pd.read_csv(TEMP_CSV, encoding="utf-8")
sampling_df["MEAS_DT"] = pd.to_datetime(sampling_df["MEAS_DT"], errors="coerce")
print(f"전체 행 수: {len(sampling_df):,}행")
print(f"시각 변환 실패: {sampling_df['MEAS_DT'].isna().sum():,}건")
print(f"측정 기간: {sampling_df['MEAS_DT'].min()} ~ {sampling_df['MEAS_DT'].max()}")

# [목적] 서로 다른 LOT 사이의 비가동 시간을 제외하고 LOT 내부 측정 간격을 계산합니다.
sampling_df = sampling_df.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
# 괄호를 사용하면 긴 계산식을 여러 줄로 나눠 읽기 쉽게 작성할 수 있습니다.
sampling_df["gap_sec"] = (
    sampling_df.groupby("LOT_NO")["MEAS_DT"].diff().dt.total_seconds()
)
sampling_gap_sec = sampling_df["gap_sec"].dropna()

# [목적] 간격별 발생 건수와 전체 LOT 내부 간격 대비 비율을 표로 만듭니다.
# [확인] 가장 건수가 많은 간격이 대표 측정 주기입니다.
sampling_summary = pd.DataFrame({
    "count": sampling_gap_sec.value_counts().sort_index(),
    "pct": (sampling_gap_sec.value_counts(normalize=True).sort_index() * 100).round(4),
})
display(sampling_summary)

# [목적] 대표 간격, 비율, 다른 간격을 자동으로 문장으로 출력합니다.
representative_sec = sampling_gap_sec.mode().iloc[0]
representative_count = int((sampling_gap_sec == representative_sec).sum())
representative_pct = (sampling_gap_sec == representative_sec).mean() * 100
other_intervals = sorted(sampling_gap_sec[sampling_gap_sec != representative_sec].unique())
print(
    f"온도 데이터는 LOT 내부에서 대부분 {representative_sec:g}초 간격으로 기록되었습니다. "
    f"{representative_count:,}건으로 전체 LOT 내부 간격의 {representative_pct:.2f}%입니다."
)
print("다른 측정 간격(초):", other_intervals)
print("중복 시각(0초 간격):", int((sampling_gap_sec == 0).sum()), "건")
print("역순 시각(음수 간격):", int((sampling_gap_sec < 0).sum()), "건")
print("최대 LOT 내부 간격:", sampling_gap_sec.max(), "초")

# [목적] 간격별 건수를 시각화합니다. 건수 차이가 커서 y축은 로그 눈금을 사용합니다.
# kind='bar'는 막대그래프, logy=True는 y축을 로그 눈금으로 지정합니다.
# 1초 건수와 3~11초 건수의 차이가 매우 크므로 일반 눈금보다 로그 눈금이 비교에 적합합니다.
ax = sampling_summary["count"].plot(
    kind="bar", figsize=(10, 4), color="steelblue", logy=True
)
ax.set_title("LOT 내부 측정 간격 분포")
ax.set_xlabel("측정 간격(초)")
ax.set_ylabel("발생 건수 (로그 눈금)")
plt.xticks(rotation=0)
# tight_layout은 제목과 축 글자가 그림 밖으로 잘리는 것을 줄여줍니다.
plt.tight_layout()
plt.show()

## 추가 코드 실행 결과

| 항목 | 확인한 값 |
| --- | --- |
| 가장 흔한 기록 간격 | 1초 |
| 1초 간격 건수 | 334,966건 |
| 1초 간격 비율 | 96.99% |
| 다른 간격 | 2초, 3초, 5초, 6초, 10초, 11초 |
| 최대 LOT 내부 간격 | 11초 |
| 전체 행 수 | 345,390행 |

> 2초 간격이 10,372건 존재합니다. 실제 누락인지 정상적인 데이터 로거 동작인지는 설비 수집 사양 확인이 필요합니다.

### 시각화 1 — LOT별 불규칙 간격 비율

**왜 추가했는가:** 전체 데이터에서 1초가 96.99%라는 결과만 보면 일부 LOT에 문제가 집중되어 있는지 알 수 없습니다. LOT별로 1초 초과 간격의 비율을 비교하면 특정 생산 구간이나 수집 세션의 문제 가능성을 찾을 수 있습니다.

**어떤 부분을 보면 좋은가:**

- 막대가 다른 LOT보다 유난히 높은 LOT가 있는지 봅니다.
- 불규칙 비율뿐 아니라 표의 `max_gap_sec`도 함께 확인합니다.
- 행 수가 작은 LOT는 비율이 쉽게 커질 수 있으므로 `interval_count`도 같이 봅니다.

**점검할 것:** 특정 LOT의 불규칙 비율이 높다면 해당 시기의 네트워크, 데이터 로거, 설비 정지 기록과 대조해야 합니다. 2초 간격이 정상 사양이라면 `> 1초` 대신 현장 허용 기준으로 조건을 바꿔야 합니다.

In [ ]:
# 같은 LOT 내부 간격을 원본 분석용 데이터에 저장합니다.
# LOT별 diff 결과를 df에 새 컬럼으로 저장해 이후 집계와 시점 조회에서 함께 사용합니다.
df["gap_sec"] = df.groupby("LOT_NO")["MEAS_DT"].diff().dt.total_seconds()

# 메서드 체이닝: 결측 제거 → LOT별 그룹 생성 → 여러 통계 집계 순서로 실행됩니다.
lot_interval_summary = (
    # 각 LOT의 첫 행은 이전 시각이 없어 간격이 NaN이므로 집계에서 제외합니다.
    df.dropna(subset=["gap_sec"])
      .groupby("LOT_NO")
      .agg(
          interval_count=("gap_sec", "size"),
          # lambda는 LOT별 간격 중 1초를 초과한 값의 개수를 계산하는 짧은 함수입니다.
          irregular_count=("gap_sec", lambda x: (x > 1).sum()),
          max_gap_sec=("gap_sec", "max"),
      )
)
# LOT 크기가 서로 다르므로 단순 건수 대신 전체 간격 수로 나눈 비율도 계산합니다.
lot_interval_summary["irregular_pct"] = (
    lot_interval_summary["irregular_count"]
    / lot_interval_summary["interval_count"] * 100
).round(3)
display(lot_interval_summary.sort_values("irregular_pct", ascending=False))

# 비율이 높은 LOT부터 왼쪽에 배치하여 우선 점검 대상을 쉽게 찾습니다.
ax = lot_interval_summary["irregular_pct"].sort_values(ascending=False).plot(
    kind="bar", figsize=(12, 4), color="darkorange"
)
ax.set_title("LOT별 1초 초과 측정 간격 비율")
ax.set_xlabel("LOT_NO")
ax.set_ylabel("1초 초과 간격 비율(%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ✅ 확인부탁드립니다
✔️ 막대가 높은 LOT에 1초 초과 간격이 상대적으로 많이 나타나는지 확인하는 그래프입니다.
1. 높은 LOT 찾기: 그래프에서 240069는 4.044%로 가장 높습니다. 다른 LOT보다 높은 이유를 살펴볼 우선 대상입니다.
2. 비율과 건수 함께 보기: LOT마다 측정 개수가 다릅니다. 비율이 높아도 간격 건수가 적을 수 있으므로 irregular_count와 interval_count를 같이 확인하세요.
3. 간격의 크기 확인: 2초가 반복되는 것인지, 5~11초처럼 긴 간격이 있는지 max_gap_sec와 간격별 건수를 봅니다. 같은 비율이라도 양상이 다릅니다.
4. 발생 시점 확인: 간격이 LOT 전체에 퍼져 있는지, 특정 시간에 몰렸는지 시점 그래프에서 확인합니다. 긴 간격의 앞뒤 센서값과 조업 이벤트도 대조합니다.

이 그래프만으로 LOT가 불량이거나 측정값이 누락됐다고 판정할 수는 없습니다. 현재 그래프의 ‘불규칙’은 단순히 gap_sec > 1이라는 분석 기준입니다. 

⚠️ 2초 간격이 정상적인 기록 방식인지 확인되면 기준을 다시 정해야 합니다.

### 시각화 2 — 불규칙 간격이 발생한 시점

**왜 추가했는가:** 같은 수의 공백이라도 한 시점에 몰려 발생한 경우와 전체 기간에 고르게 발생한 경우는 원인이 다를 수 있습니다. 발생 시각과 간격 크기를 함께 보면 일시적인 수집 장애와 반복적인 로거 특성을 구분하는 데 도움이 됩니다.

**어떤 부분을 보면 좋은가:**

- 점이 특정 시간대에 무더기로 모여 있는지 봅니다.
- 3초 이상처럼 위쪽에 있는 큰 공백을 우선 확인합니다.
- 색이 같은 점은 같은 LOT이므로 특정 LOT에 집중되는지도 확인합니다.

**점검할 것:** 큰 공백의 직전·직후 센서값, 설비 정지 여부, 조업 이벤트 시각을 대조해야 합니다. LOT 사이 공백은 이미 제외되었으므로 이 그래프의 점은 모두 LOT 내부에서 발생한 간격입니다.

In [ ]:
# loc[조건, 컬럼]으로 1초 초과 행과 필요한 세 컬럼만 선택합니다.
# copy를 사용해 원본 df와 독립된 점검용 DataFrame을 만듭니다.
irregular = df.loc[df["gap_sec"] > 1, ["MEAS_DT", "LOT_NO", "gap_sec"]].copy()
print(f"1초 초과 LOT 내부 간격: {len(irregular):,}건")
display(irregular.sort_values("gap_sec", ascending=False).head(20))

# subplots는 그래프 전체(fig)와 실제 축(ax)을 반환합니다. figsize는 가로·세로 크기입니다.
fig, ax = plt.subplots(figsize=(14, 5))
# LOT별로 반복하면서 서로 다른 범례 항목을 가진 점을 그립니다.
for lot_no, part in irregular.groupby("LOT_NO"):
    # x축은 발생 시각, y축은 간격 크기입니다. s는 점 크기, alpha는 투명도입니다.
    ax.scatter(part["MEAS_DT"], part["gap_sec"], s=10, alpha=0.55, label=str(lot_no))
ax.set_title("LOT 내부 불규칙 측정 간격의 발생 시점")
ax.set_xlabel("측정 시각")
ax.set_ylabel("측정 간격(초)")
# bbox_to_anchor로 범례를 그래프 오른쪽 밖에 배치해 데이터 점을 가리지 않게 합니다.
ax.legend(title="LOT_NO", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

✔️ LOT 내부 불규칙 측정 간격 발생 시점 그래프에서는 점의 위치(언제), 높이(몇 초), **밀집 정도(반복되는지)**를 확인하면 됩니다.

1. 긴 간격부터 확인: 그래프에서 10~11초 간격이 나타난 시점을 먼저 찾고, 바로 앞뒤의 온도·제어출력 값이 급변했는지 봅니다.
2. 한때 몰렸는지 확인: 점이 짧은 시간에 집중된다면 그 시간대의 조업 이벤트나 정비 기록과 대조합니다. 기간 전체에 반복된다면 기록 방식의 특성일 가능성도 살펴봅니다.
3. 특정 LOT에 집중됐는지 확인: 같은 LOT의 점이 유난히 많다면 해당 LOT의 시작·종료 시각, 측정 행 수, 3초 이상 간격의 건수를 함께 확인합니다.
4. LOT 경계인지 재확인: 그래프에 표시된 간격은 같은 LOT_NO 안에서 계산한 값이어야 합니다. LOT 사이의 비가동 시간이 섞이면 해석이 달라집니다.

예를 들어 이 노트북에서는 11초 간격이 3건 기록돼 있습니다. 

그 세 시점의 앞뒤 행과 이벤트를 먼저 살펴보면 좋습니다. 

다만 그래프의 점만으로 수집 누락이나 설비 이상을 확정할 수는 없습니다.

### 최종 해석 시 꼭 확인할 사항

1. **대표 간격:** 최빈값과 중앙값이 모두 1초인지 확인합니다.
2. **분모:** 전체 행 수가 아니라 LOT별 첫 행을 제외한 `345,376개` 간격을 비율의 분모로 사용했는지 확인합니다.
3. **LOT 경계:** 서로 다른 LOT 사이의 장기 공백을 센서 누락으로 계산하지 않았는지 확인합니다.
4. **중복·역순:** 0초 또는 음수 간격이 있는지 확인합니다. 현재 결과는 모두 0건입니다.
5. **2초 간격:** 10,372건이므로 단순한 우연으로 보기 어렵습니다. 수집기 사양상 허용된 동작인지 확인해야 합니다.
6. **큰 공백:** 최대 11초 간격의 원인을 원본 행과 조업 이벤트에서 확인합니다.
7. **결론 표현:** 현장 사양 확인 전에는 2초 이상을 확정적인 데이터 누락이라고 표현하지 않고 `불규칙 간격` 또는 `누락 후보`라고 기록합니다
